In [0]:
!pip install  transformers torch langdetect 


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 33.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 887.9/888.1 MB

*** WARNING: max output size exceeded, skipping output. ***

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/7

In [0]:
%restart_python

In [0]:
# ===== Simple LLM Evaluation + Contradiction Reasoning for Number Plate Dataset =====

import pandas as pd
import numpy as np
import re
from sklearn.metrics import accuracy_score, confusion_matrix
from langdetect import detect
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    pipeline
)

# ---------------------------
# 0) Setup & helpers
# ---------------------------
DATA_PATH = "Dataset.csv"   # <= change if needed
SAVE_CONTRADICTIONS_CSV = "contradiction_analysis.csv"

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

# Category phrases (for NLI hypotheses + zero-shot labels + keyword rules)
CATEGORY_PHRASES = {
    "A": "visibility issues such as faded or blurred characters",
    "B": "illegal font or style",
    "C": "missing, broken, or incomplete plate",
    "D": "wrong or non-compliant color",
    "E": "misaligned or crooked mounting",
    "F": "offensive or inappropriate language",
    "G": "fake, duplicate, or forged plate",
}

CATEGORY_KEYWORDS = {
    "A": ["faded", "blurred", "smudged", "damaged", "unreadable", "worn"],
    "B": ["illegal font", "wrong font", "stylized", "script font", "non-standard font"],
    "C": ["missing", "broken", "incomplete", "no plate", "lost"],
    "D": ["wrong color", "incorrect color", "non-compliant color", "background color"],
    "E": ["misaligned", "crooked", "tilted", "not straight", "improper mounting"],
    "F": ["offensive", "profanity", "abusive", "inappropriate", "slur"],
    "G": ["fake", "duplicate", "forged", "counterfeit", "replica"]
}

def find_matched_keywords(text):
    t = str(text).lower()
    hits = {cat: [kw for kw in kws if kw in t] for cat, kws in CATEGORY_KEYWORDS.items()}
    hits = {c: v for c, v in hits.items() if v}
    return hits

# ---------------------------
# 1) Load data
# ---------------------------
df = pd.read_csv(DATA_PATH)
# After loading data
df["description"] = df["description"].fillna("").astype(str)
df.loc[df["description"].str.strip() == "", "description"] = "No description provided."

# Category column: ensure proper handling of NaN / empty
df["category"] = df["category"].fillna("None").astype(str)
df.loc[df["category"].str.strip() == "", "category"] = "None"

# normalize columns
for col in ["classify", "category", "pred_Classify", "pred_Category"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.capitalize()

df["category"] = df["category"].replace({"Nan": np.nan})
df["category"] = df["category"].fillna("None")

# ---------------------------
# 2) Build inference pipelines
# ---------------------------
# Zero-shot for decision/category (fast reasoning baseline + auto-preds if missing)
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0 if torch.cuda.is_available() else -1)

# Toxicity scorer (optional; quick)
tox = pipeline("text-classification", model="unitary/toxic-bert", device=0 if torch.cuda.is_available() else -1)

# True NLI for entail/contradict (description vs. decision/category hypotheses)
nli_tokenizer = AutoTokenizer.from_pretrained("roberta-large-mnli")
nli_model = AutoModelForSequenceClassification.from_pretrained("roberta-large-mnli")
id2label = nli_model.config.id2label  # {0:'CONTRADICTION',1:'NEUTRAL',2:'ENTAILMENT'}

def nli_score(premise, hypothesis):
    inputs = nli_tokenizer(str(premise), str(hypothesis), return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = nli_model(**inputs).logits[0].numpy()
    probs = softmax(logits)
    # roberta-large-mnli label order is typically: 0: CONTRADICTION, 1: NEUTRAL, 2: ENTAILMENT
    idx = probs.argmax()
    return id2label[idx].lower(), float(probs[idx])

# ---------------------------
# 3) Predictions (if not provided)
# ---------------------------
if "pred_Classify" not in df.columns:
    # derive from description via zero-shot (Approve vs Reject)
    def predict_decision(desc):
        labels = ["Approve", "Reject"]
        res = zsc(str(desc), labels, hypothesis_template="This plate should be {}.")
        return res["labels"][0]
    df["pred_Classify"] = df["description"].apply(predict_decision)

if "pred_Category" not in df.columns:
    category_labels_for_zsc = [f"{k}: {v}" for k, v in CATEGORY_PHRASES.items()]
    key_map = {f"{k}: {v}": k for k, v in CATEGORY_PHRASES.items()}

    def predict_category(desc):
        if desc.strip().lower() == "no description provided.":
            return "None"
        res = zsc(desc, category_labels_for_zsc, multi_label=False)
        top = res["labels"][0]
        return key_map[top]

    df["pred_Category"] = df.apply(
        lambda r: predict_category(r["description"]) if r["pred_Classify"] == "Reject" else "None",
        axis=1
    )


# ---------------------------
# 4) Metrics
# ---------------------------
true_decision = df["classify"]
pred_decision = df["pred_Classify"]

decision_acc = accuracy_score(true_decision, pred_decision)

reject_df = df[df["classify"] == "Reject"].copy()
cat_acc = None
if len(reject_df) and "pred_Category" in reject_df.columns:
    cat_acc = accuracy_score(reject_df["category"], reject_df["pred_Category"])

# Description–Category Consistency (rule-based)
def desc_cat_consistent(row):
    if row["classify"] != "Reject":
        return True
    desc = str(row["description"]).lower()
    kws = CATEGORY_KEYWORDS.get(str(row["category"]), [])
    return any(kw in desc for kw in kws) if kws else False

df["desc_consistent"] = df.apply(desc_cat_consistent, axis=1)
desc_consistency = df["desc_consistent"].mean()

# Toxicity on plate text
df["toxicity_score"] = df["plate_number"].apply(lambda x: tox(str(x))[0]["score"])
avg_tox_approve = df[df["classify"] == "Approve"]["toxicity_score"].mean()
avg_tox_reject = df[df["classify"] == "Reject"]["toxicity_score"].mean()

# Language detection accuracy (optional)
lang_acc = None
if "language" in df.columns:
    df["pred_lang"] = df["plate_number"].apply(lambda x: detect(str(x)))
    lang_acc = accuracy_score(df["language"].astype(str).str.lower(), df["pred_lang"].astype(str).str.lower())

print(f"Decision Accuracy: {decision_acc:.2%}")
if cat_acc is not None:
    print(f"Category Accuracy (Rejects only): {cat_acc:.2%}")
print(f"Description–Category Consistency: {desc_consistency:.2%}")
print(f"Avg Toxicity (Approved): {avg_tox_approve:.3f}")
print(f"Avg Toxicity (Rejected): {avg_tox_reject:.3f}")
if lang_acc is not None:
    print(f"Language Detection Accuracy: {lang_acc:.2%}")

# ---------------------------
# 5) Contradiction reasoning (NLI + zero-shot hints)
# ---------------------------
def hypothesis_for(label, category):
    label = str(label).strip().capitalize()
    if label == "Approve":
        return "This plate is compliant and should be approved."
    # Reject
    phrase = CATEGORY_PHRASES.get(category, "").strip()
    if phrase:
        return f"This plate should be rejected due to {phrase}."
    return "This plate should be rejected."

def best_zs_category(desc):
    # Return (cat, score)
    labels = [f"{k}: {v}" for k, v in CATEGORY_PHRASES.items()]
    res = zsc(str(desc), labels, multi_label=False)
    top_label, top_score = res["labels"][0], float(res["scores"][0])
    # map back to A..G
    for k, v in CATEGORY_PHRASES.items():
        if top_label.startswith(f"{k}:"):
            return k, top_score
    # fallback
    return "A", top_score

def zs_decision(desc):
    res = zsc(str(desc), ["Approve", "Reject"], hypothesis_template="This plate should be {}.")
    return res["labels"][0], float(res["scores"][0])

contradictions = []
for _, row in df.iterrows():
    true_label = row["classify"]
    pred_label = row["pred_Classify"]

    if true_label != pred_label:
        desc = row["description"]
        true_cat = row.get("category", "None")
        pred_cat = row.get("pred_Category", "None")

        # Build hypotheses
        hyp_true = hypothesis_for(true_label, true_cat if true_label == "Reject" else "None")
        hyp_pred = hypothesis_for(pred_label, pred_cat if pred_label == "Reject" else "None")

        # NLI both ways
        nli_true_lbl, nli_true_p = nli_score(desc, hyp_true)
        nli_pred_lbl, nli_pred_p = nli_score(desc, hyp_pred)

        # Zero-shot hints
        zs_dec, zs_dec_p = zs_decision(desc)
        zs_cat, zs_cat_p = best_zs_category(desc)

        # Keyword matches
        kw_hits = find_matched_keywords(desc)
        kw_flat = "; ".join([f"{k}→{', '.join(v)}" for k, v in kw_hits.items()]) if kw_hits else ""

        # Compose a short reason
        reason = (
            f"Description {nli_true_lbl} the TRUE hypothesis ('{hyp_true}', {nli_true_p:.2f}) "
            f"and {nli_pred_lbl} the PRED hypothesis ('{hyp_pred}', {nli_pred_p:.2f}). "
            f"Zero-shot leans '{zs_dec}' ({zs_dec_p:.2f}); top category ~ {zs_cat} ({zs_cat_p:.2f}). "
            f"{'Keywords: ' + kw_flat if kw_flat else 'No category keywords found.'}"
        )

        contradictions.append({
            "plate_number": row.get("plate_number"),
            "true_decision": true_label,
            "true_category": true_cat,
            "pred_decision": pred_label,
            "pred_category": pred_cat,
            "description": desc,
            "nli_true_label": nli_true_lbl,
            "nli_true_conf": round(nli_true_p, 3),
            "nli_pred_label": nli_pred_lbl,
            "nli_pred_conf": round(nli_pred_p, 3),
            "zs_decision": zs_dec,
            "zs_decision_conf": round(zs_dec_p, 3),
            "zs_category": zs_cat,
            "zs_category_conf": round(zs_cat_p, 3),
            "matched_keywords": kw_flat,
            "reason": reason
        })

contradict_df = pd.DataFrame(contradictions)
if len(contradict_df):
    contradict_df.to_csv(SAVE_CONTRADICTIONS_CSV, index=False)
    print(f"\nSaved contradiction analysis to: {SAVE_CONTRADICTIONS_CSV}")
    # Show a quick peek
    with pd.option_context("display.max_colwidth", 120):
        print("\nSample contradictions with reasoning:\n", contradict_df.head(10))
else:
    print("\nNo decision contradictions found between ground truth and predictions.")

# ---------------------------
# 6) (Optional) Confusion Matrices — numbers only (no plotting)
# ---------------------------
labels_dec = ["Approve", "Reject"]
cm_dec = confusion_matrix(true_decision, pred_decision, labels=labels_dec)
print("\nDecision Confusion Matrix (rows=true, cols=pred):")
print(pd.DataFrame(cm_dec, index=labels_dec, columns=labels_dec))

if len(reject_df) and "pred_Category" in reject_df.columns:
    cats = list(CATEGORY_PHRASES.keys())
    cm_cat = confusion_matrix(reject_df["category"], reject_df["pred_Category"], labels=cats)
    print("\nCategory Confusion Matrix (Rejects only; rows=true, cols=pred):")
    print(pd.DataFrame(cm_cat, index=cats, columns=cats))

# ---------------------------
# 7) Summary
# ---------------------------
summary = {
    "Decision Accuracy": decision_acc,
    "Category Accuracy (Rejects only)": cat_acc,
    "Desc–Category Consistency": desc_consistency,
    "Avg Toxicity Approve": avg_tox_approve,
    "Avg Toxicity Reject": avg_tox_reject,
    "Language Detection Accuracy": lang_acc
}
print("\n===== METRICS SUMMARY =====")
for k, v in summary.items():
    if v is None:
        continue
    if k.endswith("Accuracy") or "Consistency" in k:
        print(f"{k}: {v:.2%}")
    else:
        print(f"{k}: {v:.3f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Decision Accuracy: 75.76%
Category Accuracy (Rejects only): 13.51%
Description–Category Consistency: 26.26%
Avg Toxicity (Approved): 0.033
Avg Toxicity (Rejected): 0.021
Language Detection Accuracy: 0.00%

Saved contradiction analysis to: contradiction_analysis.csv

Sample contradictions with reasoning:
   plate_number  ...                                                                                                                   reason
0     rehmat20  ...  Description neutral the TRUE hypothesis ('This plate should be rejected due to missing, broken, or incomplete plate....
1     rvrhghts  ...  Description contradiction the TRUE hypothesis ('This plate should be rejected due to missing, broken, or incomplete ...
2       SAIIMA  ...  Description contradiction the TRUE hypothesis ('This plate should be rejected due to missing, broken, or incomplete ...
3     1963merc  ...  Description neutral the TRUE hypothesis ('This plate should be rejected due to offensive or inappropriate lan